In [ ]:
import torch
import tiktoken

torch.Size([50257, 256])


In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [13]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [18]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [19]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"토큰 ID:\n{inputs}\n")
print(f"입력 크기:\n{inputs.shape}")

토큰 ID:
tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

입력 크기:
torch.Size([8, 4])


In [22]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(token_embedding_layer.weight.shape)

token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape) # torch.Size([8, 4, 256])

torch.Size([50257, 256])
torch.Size([8, 4, 256])


In [29]:
context_length = max_length

pos_embedding_layer = torch.nn.Embedding(4, 256)
#   -> 0,1,2,3 네 개 자리 각각에 256차원 벡터를 담은 표(4 x 256)를 준비

pos_embeddings = pos_embedding_layer(torch.arange(4))
#   torch.arange(4) -> tensor([0, 1, 2, 3])
#   -> 0,1,2,3 각 위치 ID로 표를 룩업
#   -> 0번,1번,2번,3번 자리에 대한 256차원 벡터 4개를 꺼냄

# 토큰은 문장마다 내용이 바뀌지만, "자리 번호"는 어떤 문장이 오든 언제나 0,1,2,3이죠. 그래서 위치 임베딩에는 항상 torch.arange(context_length)라는 고정된 순번을 넣어주는 겁니다. "몇 번째 자리인가"는 문장 내용과 무관하니까요.

print(pos_embeddings)
print(pos_embeddings.shape) # torch.Size([4, 256])

tensor([[-1.2324, -0.0286, -1.5265,  ..., -0.7801, -1.7978, -0.0773],
        [-0.2347,  0.7412,  0.7203,  ...,  0.6111, -0.7801, -0.3734],
        [-0.3792, -1.2898, -1.7925,  ..., -2.2316, -0.7608, -0.5647],
        [ 0.5782, -0.2117,  0.4530,  ...,  0.6547,  0.5555, -1.5536]],
       grad_fn=<EmbeddingBackward0>)
torch.Size([4, 256])


In [31]:
input_embeddings = token_embeddings + pos_embeddings
print(token_embeddings.shape)
print(pos_embeddings.shape)
print(input_embeddings.shape)

torch.Size([8, 4, 256])
torch.Size([4, 256])
torch.Size([8, 4, 256])
